# Ethical Web Scraping Guidelines

This notebook follows these ethical 'rules of the road':
- **Rate Limiting**: A delay of 1.5 seconds is set between requests to ensure we don't impact the server's performance for other users.
- **User-Agent Identification**: The scraper identifies itself using a custom User-Agent string (`CricinfoDataProject/1.0`).
- **Data Minimization**: Only the necessary scorecard data for the specified period is scraped.
- **No Redistribution**: The data is for personal project use only and should not be re-published.

In [1]:
import time
try:
  import requests
  from bs4 import BeautifulSoup
  import pandas as pd
except:
  !pip3 install requests
  !pip3 install beautifulsoup4
  !pip3 install pandas
  import requests
  from bs4 import BeautifulSoup
  import pandas as pd


# Match-By-Match Batting Stats Scraping

In [ ]:
HEADERS = {'User-Agent': 'CricinfoDataProject/1.0 (contact: arkosaha.ruet@gmail.com)'}
# Define the base URL and the initial page URL
base_url = 'https://stats.espncricinfo.com'
url = base_url + '/ci/engine/stats/index.html?class=3;page=1;spanmin1=1+Jan+2022;spanval1=span;template=results;type=batting;view=innings'

response = requests.get(url, headers=HEADERS)
response = response.content
soup = BeautifulSoup(response, 'html.parser')


In [3]:
# Initialize the page number
page_number = 1
total_pages = int(soup.find_all('td', class_='left')[3].text.split(' ')[6].rstrip())


In [4]:
page_number


1

In [5]:
total_pages


1013

In [6]:
data = []
for i in range(1, total_pages+1):
  url = f"https://stats.espncricinfo.com/ci/engine/stats/index.html?class=3;page={i};spanmin1=1+Jan+2022;spanval1=span;template=results;type=batting;view=innings"
  time.sleep(1.5)
  response = requests.get(url, headers=HEADERS)
  response = response.content
  soup = BeautifulSoup(response, 'html.parser')
  # Find the table element that contains the data we want to extract
  table = soup.select_one('#ciHomeContentlhs > div.pnl650M > table:nth-child(5)')

  # Check if the table exists
  if table is not None:
    # Loop through the rows of the table and extract the data for each player
    for row in table.tbody.find_all('tr'):
      columns = row.find_all('td')
      if len(columns) > 0:
        Player = columns[0].text.strip()
        Runs = columns[1].text.strip()
        Mins = columns[2].text.strip()
        BF = columns[3].text.strip()
        Fours = columns[4].text.strip()
        Sixes = columns[5].text.strip()
        SR = columns[6].text.strip()
        Inns = columns[7].text.strip()
        Opposition = columns[9].text.strip()
        Ground = columns[10].text.strip()
        Start_Date = columns[11].text.strip()

        # Check if the Runs column has a '*'
        if '*' in Runs:
          Not_Out = 1
        else:
          Not_Out = 0

        # Trim the asterisk (*) from the Runs column
        Runs = Runs.strip('*')

        # Add the data for each player to the DataFrame
        data.append({
          'Player': Player,
          'Runs': Runs,
          'Mins': Mins,
          'Not_Out': Not_Out,
          'BF': BF,
          'Fours': Fours,
          'Sixes': Sixes,
          'SR': SR,
          'Inns': Inns,
          'Opposition': Opposition,
          'Ground': Ground,
          'Start_Date': Start_Date
        })

    # Check if there is a next page by finding the 'Next' button on the page
    next_page_link = soup.select_one('.PaginationLink')
    if next_page_link is not None:
      # If there is a next page, increment the page number and update the URL to the next page
      i += 1
    else:
      # If there is no next page, break out of the loop
      break

# Create a DataFrame from the scraped data
df = pd.DataFrame(data)


In [7]:
# Print the DataFrame
df.head(10)


,Player,Runs,Mins,Not_Out,BF,Fours,Sixes,SR,Inns,Opposition,Ground,Start_Date
0,Mohammad Ihsan (ESP),160,-,0,63,5,17,253.96,1,v Croatia,Cartagena,7 Dec 2025
1,YSD Seneveratne (CAY),150,-,1,67,7,13,223.88,1,v Brazil,Buenos Aires,14 Dec 2024
2,Sahil Chauhan (EST),144,-,1,41,6,18,351.21,2,v Cyprus,Episkopi,17 Jun 2024
3,PD Salt (ENG),141,101,1,60,15,8,235,1,v South Africa,Manchester,12 Sep 2025
4,Kushal Malla (NEP),137,-,1,50,8,12,274,1,v Mongolia,Hangzhou,27 Sep 2023
5,Zeeshan Kukikhel (HUN),137,-,0,49,7,15,279.59,2,v Austria,Lower Austria,5 Jun 2022
6,FH Allen (NZ),137,94,0,62,5,16,220.96,1,v Pakistan,Dunedin,17 Jan 2024
7,M Levitt (NED),135,-,0,62,11,10,217.74,1,v Namibia,Kirtipur,29 Feb 2024
8,Abhishek Sharma (IND),135,93,0,54,7,13,250,1,v England,Wankhede,2 Feb 2025
9,L Yamamoto-Lake (JPN),134,-,1,68,8,12,197.05,1,v China,Mong Kok,15 Feb 2024


In [8]:
# Trim the v before the country name in the opposition column
df['Opposition'] = df['Opposition'].str.lstrip('v ')


In [9]:
# Extract player name and country using regular expression
df[['Player_Name', 'Country']] = df['Player'].str.extract(r'^(.*?) \((.*?)\)')


In [10]:
df


,Player,Runs,Mins,Not_Out,BF,Fours,Sixes,SR,Inns,Opposition,Ground,Start_Date,Player_Name,Country
0,Mohammad Ihsan (ESP),160,-,0,63,5,17,253.96,1,Croatia,Cartagena,7 Dec 2025,Mohammad Ihsan,ESP
1,YSD Seneveratne (CAY),150,-,1,67,7,13,223.88,1,Brazil,Buenos Aires,14 Dec 2024,YSD Seneveratne,CAY
2,Sahil Chauhan (EST),144,-,1,41,6,18,351.21,2,Cyprus,Episkopi,17 Jun 2024,Sahil Chauhan,EST
3,PD Salt (ENG),141,101,1,60,15,8,235,1,South Africa,Manchester,12 Sep 2025,PD Salt,ENG
4,Kushal Malla (NEP),137,-,1,50,8,12,274,1,Mongolia,Hangzhou,27 Sep 2023,Kushal Malla,NEP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50625,Arshdeep Singh (IND),DNB,-,0,-,-,-,-,1,England,Wankhede,5 Mar 2026,Arshdeep Singh,IND
50626,LA Dawson (ENG),DNB,-,0,-,-,-,-,2,India,Wankhede,5 Mar 2026,LA Dawson,ENG
50627,AU Rashid (ENG),DNB,-,0,-,-,-,-,2,India,Wankhede,5 Mar 2026,AU Rashid,ENG
50628,Azri Azhar (MAS),DNB,-,0,-,-,-,-,1,Bahrain,Kuala Lumpur,7 Mar 2026,Azri Azhar,MAS


In [11]:
# Reorder columns
df = df[['Player_Name', 'Country', 'Runs', 'Not_Out', 'BF', 'Fours', 'Sixes', 'SR', 'Inns', 'Opposition', 'Ground', 'Start_Date']]


In [14]:
def replace_values(value):
    if value == 'DNB':
      return 0
    elif value == 'TDNB':
      return 0
    elif value == 'absent':
      return 0
    elif value == 'sub':
      return 0
    elif value == '-':
      return 0
    else:
      return value

# apply the replace_values function to all columns in the DataFrame
df = df.map(replace_values)
df


,Player_Name,Country,Runs,Not_Out,BF,Fours,Sixes,SR,Inns,Opposition,Ground,Start_Date
0,Mohammad Ihsan,ESP,160,0,63,5,17,253.96,1,Croatia,Cartagena,7 Dec 2025
1,YSD Seneveratne,CAY,150,1,67,7,13,223.88,1,Brazil,Buenos Aires,14 Dec 2024
2,Sahil Chauhan,EST,144,1,41,6,18,351.21,2,Cyprus,Episkopi,17 Jun 2024
3,PD Salt,ENG,141,1,60,15,8,235,1,South Africa,Manchester,12 Sep 2025
4,Kushal Malla,NEP,137,1,50,8,12,274,1,Mongolia,Hangzhou,27 Sep 2023
...,...,...,...,...,...,...,...,...,...,...,...,...
50625,Arshdeep Singh,IND,0,0,0,0,0,0,1,England,Wankhede,5 Mar 2026
50626,LA Dawson,ENG,0,0,0,0,0,0,2,India,Wankhede,5 Mar 2026
50627,AU Rashid,ENG,0,0,0,0,0,0,2,India,Wankhede,5 Mar 2026
50628,Azri Azhar,MAS,0,0,0,0,0,0,1,Bahrain,Kuala Lumpur,7 Mar 2026


In [15]:
df.to_csv('data/batters.csv')


# Match-By-Match Bowling Stats Scraping

In [16]:
# Define the base URL and the initial page URL
base_url = 'https://stats.espncricinfo.com'
url = base_url + '/ci/engine/stats/index.html?class=3;page=1;spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling;view=innings'

response = requests.get(url, headers=HEADERS)
response = response.content
soup = BeautifulSoup(response, 'html.parser')


In [17]:
# Initialize the page number
page_number = 1
total_pages = int(soup.find_all('td', class_='left')[3].text.split(' ')[6].rstrip())


In [18]:
data2 = []
for i in range(1, total_pages+1):
  url = f"https://stats.espncricinfo.com/ci/engine/stats/index.html?class=3;page={i};spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling;view=innings"
  time.sleep(1.5)
  response = requests.get(url, headers=HEADERS)
  response = response.content
  soup = BeautifulSoup(response, 'html.parser')
  # Find the table element that contains the data we want to extract
  table = soup.select_one('#ciHomeContentlhs > div.pnl650M > table:nth-child(5)')

  # Check if the table exists
  if table is not None:
    # Loop through the rows of the table and extract the data for each player
    for row in table.tbody.find_all('tr'):
      columns = row.find_all('td')
      if len(columns) > 0:
        Player = columns[0].text.strip()
        Overs = columns[1].text.strip()
        Maidens = columns[2].text.strip()
        Runs = columns[3].text.strip()
        Wickets = columns[4].text.strip()
        Economy = columns[5].text.strip()
        Inns = columns[6].text.strip()
        Opposition = columns[8].text.strip()
        Ground = columns[9].text.strip()
        Start_Date = columns[10].text.strip()

        # Add the data for each player to the DataFrame
        data2.append({
          'Player': Player,
          'Overs': Overs,
          'Maidens': Maidens,
          'Runs': Runs,
          'Wickets': Wickets,
          'Economy': Economy,
          'Inns': Inns,
          'Opposition': Opposition,
          'Ground': Ground,
          'Start_Date': Start_Date
        })

    # Check if there is a next page by finding the 'Next' button on the page
    next_page_link = soup.select_one('.PaginationLink')
    if next_page_link is not None:
      # If there is a next page, increment the page number and update the URL to the next page
      i += 1
    else:
      # If there is no next page, break out of the loop
      break

# Create a DataFrame from the scraped data
df2 = pd.DataFrame(data2)


In [19]:
# Print the DataFrame
df2.head(10)


,Player,Overs,Maidens,Runs,Wickets,Economy,Inns,Opposition,Ground,Start_Date
0,S Yeshey (BHU),4.0,1,7,8,1.75,2,v Myanmar,Gelephu,26 Dec 2025
1,Syazrul Idrus (MAS),4.0,1,8,7,2,1,v China,Kuala Lumpur,26 Jul 2023
2,Ali Dawood (BHR),4.0,0,19,7,4.75,2,v Bhutan,Gelephu,11 Dec 2025
3,H Bharadwaj (SGP),4.0,2,3,6,0.75,1,v Mongolia,Bangi,5 Sep 2024
4,MA Baig (MWI),4.0,0,9,6,2.25,2,v Rwanda,Benoni,16 Dec 2023
5,Rizwan Butt (BHR),3.5,1,9,6,2.34,1,v Malawi,Blantyre,12 Jul 2025
6,JJ Smit (NAM),4.0,1,10,6,2.5,2,v Uganda,Windhoek,10 Apr 2022
7,Mustafizur Rahman (BAN),4.0,1,10,6,2.5,1,v U.S.A.,Prairie View,25 May 2024
8,A Bohara (NEP),3.4,0,11,6,3,2,v Maldives,Hangzhou,1 Oct 2023
9,Nasrulla Rana (HKG),4.0,1,12,6,3,2,v P.N.G.,Kuala Lumpur,24 Sep 2023


In [20]:
# Trim the v before the country name in the opposition column
df2['Opposition'] = df2['Opposition'].str.lstrip('v ')


In [21]:
# Extract player name and country using regular expression
df2[['Player_Name', 'Country']] = df2['Player'].str.extract(r'^(.*?) \((.*?)\)')


In [22]:
df2


,Player,Overs,Maidens,Runs,Wickets,Economy,Inns,Opposition,Ground,Start_Date,Player_Name,Country
0,S Yeshey (BHU),4.0,1,7,8,1.75,2,Myanmar,Gelephu,26 Dec 2025,S Yeshey,BHU
1,Syazrul Idrus (MAS),4.0,1,8,7,2,1,China,Kuala Lumpur,26 Jul 2023,Syazrul Idrus,MAS
2,Ali Dawood (BHR),4.0,0,19,7,4.75,2,Bhutan,Gelephu,11 Dec 2025,Ali Dawood,BHR
3,H Bharadwaj (SGP),4.0,2,3,6,0.75,1,Mongolia,Bangi,5 Sep 2024,H Bharadwaj,SGP
4,MA Baig (MWI),4.0,0,9,6,2.25,2,Rwanda,Benoni,16 Dec 2023,MA Baig,MWI
...,...,...,...,...,...,...,...,...,...,...,...,...
50625,Ahmed Faiz (MAS),DNB,-,-,-,-,2,Bahrain,Kuala Lumpur,7 Mar 2026,Ahmed Faiz,MAS
50626,Muhamad Syahadat (MAS),DNB,-,-,-,-,2,Bahrain,Kuala Lumpur,7 Mar 2026,Muhamad Syahadat,MAS
50627,Zubaidi Zulkifle (MAS),DNB,-,-,-,-,2,Bahrain,Kuala Lumpur,7 Mar 2026,Zubaidi Zulkifle,MAS
50628,Aslam Khan (MAS),DNB,-,-,-,-,2,Bahrain,Kuala Lumpur,7 Mar 2026,Aslam Khan,MAS


In [23]:
# Reorder columns
df2 = df2[['Player_Name', 'Country', 'Overs', 'Maidens', 'Runs', 'Wickets', 'Economy', 'Inns', 'Opposition', 'Ground', 'Start_Date']]


In [24]:
df2


,Player_Name,Country,Overs,Maidens,Runs,Wickets,Economy,Inns,Opposition,Ground,Start_Date
0,S Yeshey,BHU,4.0,1,7,8,1.75,2,Myanmar,Gelephu,26 Dec 2025
1,Syazrul Idrus,MAS,4.0,1,8,7,2,1,China,Kuala Lumpur,26 Jul 2023
2,Ali Dawood,BHR,4.0,0,19,7,4.75,2,Bhutan,Gelephu,11 Dec 2025
3,H Bharadwaj,SGP,4.0,2,3,6,0.75,1,Mongolia,Bangi,5 Sep 2024
4,MA Baig,MWI,4.0,0,9,6,2.25,2,Rwanda,Benoni,16 Dec 2023
...,...,...,...,...,...,...,...,...,...,...,...
50625,Ahmed Faiz,MAS,DNB,-,-,-,-,2,Bahrain,Kuala Lumpur,7 Mar 2026
50626,Muhamad Syahadat,MAS,DNB,-,-,-,-,2,Bahrain,Kuala Lumpur,7 Mar 2026
50627,Zubaidi Zulkifle,MAS,DNB,-,-,-,-,2,Bahrain,Kuala Lumpur,7 Mar 2026
50628,Aslam Khan,MAS,DNB,-,-,-,-,2,Bahrain,Kuala Lumpur,7 Mar 2026


In [ ]:
df2.to_csv('data/bowlers.csv')


# Overall Batting Stats

In [4]:
HEADERS = {'User-Agent': 'CricinfoDataProject/1.0 (contact: arkosaha.ruet@gmail.com)'}
# Define the initial page URL
url = 'https://stats.espncricinfo.com/ci/engine/stats/index.html?class=3;filter=advanced;orderby=runs;page=1;size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=batting'

response = requests.get(url, headers=HEADERS)
response = response.content
soup = BeautifulSoup(response, 'html.parser')


In [5]:
# Initialize the page number
page_number = 1
total_pages = int(soup.find_all('td', class_='left')[3].text.split(' ')[6].rstrip())


In [6]:
page_number


1

In [7]:
total_pages


17

In [8]:
data = []
for i in range(1, total_pages+1):
  url = f"https://stats.espncricinfo.com/ci/engine/stats/index.html?class=3;filter=advanced;orderby=runs;page={i};size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=batting"
  time.sleep(1.5)
  response = requests.get(url, headers=HEADERS)
  response = response.content
  soup = BeautifulSoup(response, 'html.parser')
  # Find the table element that contains the data we want to extract
  table = soup.select_one('#ciHomeContentlhs > div.pnl650M > table:nth-child(5)')

  # Check if the table exists
  if table is not None:
    # Loop through the rows of the table and extract the data for each player
    for row in table.tbody.find_all('tr'):
      columns = row.find_all('td')
      if len(columns) > 0:
        Player = columns[0].text.strip()
        Span = columns[1].text.strip()
        Matches = columns[2].text.strip()
        Innings = columns[3].text.strip()
        Not_Out = columns[4].text.strip()
        Runs = columns[5].text.strip()
        Highest_Score = columns[6].text.strip()
        Average = columns[7].text.strip()
        Balls_Faced = columns[8].text.strip()
        SR = columns[9].text.strip()
        Hundreds = columns[10].text.strip()
        Fifties = columns[11].text.strip()
        Zeros = columns[12].text.strip()
        Fours = columns[13].text.strip()
        Sixes = columns[14].text.strip()


        # Add the data for each player to the DataFrame
        data.append({
          'Player': Player,
          'Span': Span,
          'Matches': Matches,
          'Innings': Innings,
          'Not_Out' : Not_Out,
          'Runs': Runs,
          'Highest_Score': Highest_Score,
          'Average' : Average,
          'Balls_Faced': Balls_Faced,
          'SR': SR,
          'Hundreds': Hundreds,
          'Fifties': Fifties,
          'Zeros' : Zeros,
          'Fours': Fours,
          'Sixes': Sixes
        })

    # Check if there is a next page by finding the 'Next' button on the page
    next_page_link = soup.select_one('.PaginationLink')
    if next_page_link is not None:
      # If there is a next page, increment the page number and update the URL to the next page
      i += 1
    else:
      # If there is no next page, break out of the loop
      break

# Create a DataFrame from the scraped data
df3 = pd.DataFrame(data)


In [9]:
# Print the DataFrame
df3.head(10)


,Player,Span,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,Muhammad Waseem (UAE),2022-2026,93,93,5,3191,112,36.26,2113,151.01,2,26,5,262,187
1,SA Yadav (IND),2022-2026,101,97,15,3028,117,36.92,1850,163.67,4,22,5,272,167
2,Sikandar Raza (ZIM),2022-2026,91,88,11,2565,133*,33.31,1766,145.24,1,16,7,192,137
3,Virandeep Singh (MAS),2022-2026,87,80,19,2496,116*,40.91,1890,132.06,1,20,6,209,107
4,P Nissanka (SL),2022-2026,79,79,5,2346,107,31.7,1800,130.33,2,16,7,243,69
5,Syed Aziz (MAS),2022-2026,91,86,14,2135,126,29.65,1485,143.77,1,14,6,181,119
6,BKG Mendis (SL),2022-2026,77,77,6,2084,86,29.35,1599,130.33,0,15,5,192,74
7,Babar Azam (PAK),2022-2026,72,68,8,1976,110*,32.93,1561,126.58,2,14,7,205,40
8,Sohail Ahmed (BHR),2022-2026,77,72,28,1901,80*,43.2,1595,119.18,0,15,6,123,80
9,JC Buttler (ENG),2022-2026,67,63,5,1897,96,32.7,1216,156,0,13,6,189,85


In [10]:
df3 = df3.drop('Span', axis=1)
df3


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,Muhammad Waseem (UAE),93,93,5,3191,112,36.26,2113,151.01,2,26,5,262,187
1,SA Yadav (IND),101,97,15,3028,117,36.92,1850,163.67,4,22,5,272,167
2,Sikandar Raza (ZIM),91,88,11,2565,133*,33.31,1766,145.24,1,16,7,192,137
3,Virandeep Singh (MAS),87,80,19,2496,116*,40.91,1890,132.06,1,20,6,209,107
4,P Nissanka (SL),79,79,5,2346,107,31.7,1800,130.33,2,16,7,243,69
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3322,Yash Talati (KENYA),2,-,-,-,-,-,-,-,-,-,-,-,-
3323,Ye Ko Ko (MYAN),1,-,-,-,-,-,-,-,-,-,-,-,-
3324,Zeeshan Abbas (BHR),3,-,-,-,-,-,-,-,-,-,-,-,-
3325,Ziaur Rahman (AFG),5,-,-,-,-,-,-,-,-,-,-,-,-


In [12]:
def replace_values(value):
    if value == 'DNB':
      return 0
    elif value == 'TDNB':
      return 0
    elif value == 'absent':
      return 0
    elif value == 'sub':
      return 0
    elif value == '-':
      return 0
    else:
      return value

# apply the replace_values function to all columns in the DataFrame
df3 = df3.map(replace_values)
df3


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,Muhammad Waseem (UAE),93,93,5,3191,112,36.26,2113,151.01,2,26,5,262,187
1,SA Yadav (IND),101,97,15,3028,117,36.92,1850,163.67,4,22,5,272,167
2,Sikandar Raza (ZIM),91,88,11,2565,133*,33.31,1766,145.24,1,16,7,192,137
3,Virandeep Singh (MAS),87,80,19,2496,116*,40.91,1890,132.06,1,20,6,209,107
4,P Nissanka (SL),79,79,5,2346,107,31.7,1800,130.33,2,16,7,243,69
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3322,Yash Talati (KENYA),2,0,0,0,0,0,0,0,0,0,0,0,0
3323,Ye Ko Ko (MYAN),1,0,0,0,0,0,0,0,0,0,0,0,0
3324,Zeeshan Abbas (BHR),3,0,0,0,0,0,0,0,0,0,0,0,0
3325,Ziaur Rahman (AFG),5,0,0,0,0,0,0,0,0,0,0,0,0


In [13]:
# Extract player name and country using regular expression
df3[['Player_Name', 'Country']] = df3['Player'].str.extract(r'^(.*?) \((.*?)\)')


In [14]:
df3


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes,Player_Name,Country
0,Muhammad Waseem (UAE),93,93,5,3191,112,36.26,2113,151.01,2,26,5,262,187,Muhammad Waseem,UAE
1,SA Yadav (IND),101,97,15,3028,117,36.92,1850,163.67,4,22,5,272,167,SA Yadav,IND
2,Sikandar Raza (ZIM),91,88,11,2565,133*,33.31,1766,145.24,1,16,7,192,137,Sikandar Raza,ZIM
3,Virandeep Singh (MAS),87,80,19,2496,116*,40.91,1890,132.06,1,20,6,209,107,Virandeep Singh,MAS
4,P Nissanka (SL),79,79,5,2346,107,31.7,1800,130.33,2,16,7,243,69,P Nissanka,SL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3322,Yash Talati (KENYA),2,0,0,0,0,0,0,0,0,0,0,0,0,Yash Talati,KENYA
3323,Ye Ko Ko (MYAN),1,0,0,0,0,0,0,0,0,0,0,0,0,Ye Ko Ko,MYAN
3324,Zeeshan Abbas (BHR),3,0,0,0,0,0,0,0,0,0,0,0,0,Zeeshan Abbas,BHR
3325,Ziaur Rahman (AFG),5,0,0,0,0,0,0,0,0,0,0,0,0,Ziaur Rahman,AFG


In [15]:
df3.to_csv('Overall_batters.csv')


# Overall Bowling Stats

In [16]:
HEADERS = {'User-Agent': 'CricinfoDataProject/1.0 (contact: arkosaha.ruet@gmail.com)'}
# Define the initial page URL
url = 'https://stats.espncricinfo.com/ci/engine/stats/index.html?class=3;filter=advanced;orderby=wickets;page=1;size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling'

response = requests.get(url, headers=HEADERS)
response = response.content
soup = BeautifulSoup(response, 'html.parser')


In [17]:
# Initialize the page number
page_number = 1
total_pages = int(soup.find_all('td', class_='left')[3].text.split(' ')[6].rstrip())


In [18]:
page_number


1

In [19]:
total_pages


17

In [20]:
data = []
for i in range(1, total_pages+1):
  url = f"https://stats.espncricinfo.com/ci/engine/stats/index.html?class=3;filter=advanced;orderby=wickets;page={i};size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling"
  time.sleep(1.5)
  response = requests.get(url, headers=HEADERS)
  response = response.content
  soup = BeautifulSoup(response, 'html.parser')
  # Find the table element that contains the data we want to extract
  table = soup.select_one('#ciHomeContentlhs > div.pnl650M > table:nth-child(5)')

  # Check if the table exists
  if table is not None:
    # Loop through the rows of the table and extract the data for each player
    for row in table.tbody.find_all('tr'):
      columns = row.find_all('td')
      if len(columns) > 0:
        Player = columns[0].text.strip()
        Span = columns[1].text.strip()
        Matches = columns[2].text.strip()
        Innings = columns[3].text.strip()
        Overs = columns[4].text.strip()
        Maidens = columns[5].text.strip()
        Runs = columns[6].text.strip()
        Wickets = columns[7].text.strip()
        BBI = columns[8].text.strip()
        Average = columns[9].text.strip()
        Economy = columns[10].text.strip()
        SR = columns[11].text.strip()
        Four_Wickets = columns[12].text.strip()
        Five_Wickets = columns[13].text.strip()


        # Add the data for each player to the DataFrame
        data.append({
          'Player': Player,
          'Span': Span,
          'Matches': Matches,
          'Innings': Innings,
          'Overs' : Overs,
          'Maidens': Maidens,
          'Runs': Runs,
          'Wickets' : Wickets,
          'BBI': BBI,
          'Average': Average,
          'Economy': Economy,
          'SR': SR,
          'Four_Wickets' : Four_Wickets,
          'Five_Wickets': Five_Wickets
        })

    # Check if there is a next page by finding the 'Next' button on the page
    next_page_link = soup.select_one('.PaginationLink')
    if next_page_link is not None:
      # If there is a next page, increment the page number and update the URL to the next page
      i += 1
    else:
      # If there is no next page, break out of the loop
      break

# Create a DataFrame from the scraped data
df4 = pd.DataFrame(data)


In [21]:
# Print the DataFrame
df4.head(10)


,Player,Span,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets
0,Rizwan Butt (BHR),2022-2026,91,90,324.5,8,2207,137,6/9,16.1,6.79,14.2,3,4
1,Arshdeep Singh (IND),2022-2026,83,82,289.2,2,2470,127,5/51,19.44,8.53,13.6,2,1
2,Ali Dawood (BHR),2023-2026,75,75,266.4,8,1656,116,7/19,14.27,6.21,13.7,6,1
3,Ehsan Khan (HKG),2022-2026,79,77,283.1,5,1771,113,4/5,15.67,6.25,15,5,0
4,Junaid Siddique (UAE),2022-2026,77,77,290.1,2,2273,108,5/35,21.04,7.83,16.1,4,1
5,Virandeep Singh (MAS),2022-2026,87,80,259.1,8,1390,106,4/5,13.11,5.36,14.6,4,0
6,H Ssenyondo (UGA),2022-2025,70,68,242.2,10,1266,103,5/8,12.29,5.22,14.1,6,1
7,PW Hasaranga (SL),2022-2026,62,62,237.2,2,1760,102,4/15,17.25,7.41,13.9,4,0
8,AR Ramjani (UGA),2022-2025,65,63,223.1,10,1070,102,5/17,10.49,4.79,13.1,4,1
9,R Ngarava (ZIM),2022-2026,75,73,260.4,8,1887,97,4/16,19.45,7.23,16.1,2,0


In [22]:
# Extract player name and country using regular expression
df4[['Player_Name', 'Country']] = df4['Player'].str.extract(r'^(.*?) \((.*?)\)')


In [23]:
df4


,Player,Span,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Rizwan Butt (BHR),2022-2026,91,90,324.5,8,2207,137,6/9,16.1,6.79,14.2,3,4,Rizwan Butt,BHR
1,Arshdeep Singh (IND),2022-2026,83,82,289.2,2,2470,127,5/51,19.44,8.53,13.6,2,1,Arshdeep Singh,IND
2,Ali Dawood (BHR),2023-2026,75,75,266.4,8,1656,116,7/19,14.27,6.21,13.7,6,1,Ali Dawood,BHR
3,Ehsan Khan (HKG),2022-2026,79,77,283.1,5,1771,113,4/5,15.67,6.25,15,5,0,Ehsan Khan,HKG
4,Junaid Siddique (UAE),2022-2026,77,77,290.1,2,2273,108,5/35,21.04,7.83,16.1,4,1,Junaid Siddique,UAE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3322,Zubaidi Zulkifle (MAS),2022-2026,65,-,-,-,-,-,-,-,-,-,-,-,Zubaidi Zulkifle,MAS
3323,Zubair Ali (QAT),2025-2026,14,-,-,-,-,-,-,-,-,-,-,-,Zubair Ali,QAT
3324,Zuhair Muhammad (KSA),2024-2024,3,-,-,-,-,-,-,-,-,-,-,-,Zuhair Muhammad,KSA
3325,L Zulu (SWZ),2022-2022,4,-,-,-,-,-,-,-,-,-,-,-,L Zulu,SWZ


In [24]:
df4 = df4.drop('Span', axis=1)
df4


,Player,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Rizwan Butt (BHR),91,90,324.5,8,2207,137,6/9,16.1,6.79,14.2,3,4,Rizwan Butt,BHR
1,Arshdeep Singh (IND),83,82,289.2,2,2470,127,5/51,19.44,8.53,13.6,2,1,Arshdeep Singh,IND
2,Ali Dawood (BHR),75,75,266.4,8,1656,116,7/19,14.27,6.21,13.7,6,1,Ali Dawood,BHR
3,Ehsan Khan (HKG),79,77,283.1,5,1771,113,4/5,15.67,6.25,15,5,0,Ehsan Khan,HKG
4,Junaid Siddique (UAE),77,77,290.1,2,2273,108,5/35,21.04,7.83,16.1,4,1,Junaid Siddique,UAE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3322,Zubaidi Zulkifle (MAS),65,-,-,-,-,-,-,-,-,-,-,-,Zubaidi Zulkifle,MAS
3323,Zubair Ali (QAT),14,-,-,-,-,-,-,-,-,-,-,-,Zubair Ali,QAT
3324,Zuhair Muhammad (KSA),3,-,-,-,-,-,-,-,-,-,-,-,Zuhair Muhammad,KSA
3325,L Zulu (SWZ),4,-,-,-,-,-,-,-,-,-,-,-,L Zulu,SWZ


In [25]:
def replace_values(value):
    if value == 'DNB':
      return 0
    elif value == 'TDNB':
      return 0
    elif value == 'absent':
      return 0
    elif value == 'sub':
      return 0
    elif value == '-':
      return 0
    else:
      return value

# apply the replace_values function to all columns in the DataFrame
df4 = df4.map(replace_values)
df4


,Player,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Rizwan Butt (BHR),91,90,324.5,8,2207,137,6/9,16.1,6.79,14.2,3,4,Rizwan Butt,BHR
1,Arshdeep Singh (IND),83,82,289.2,2,2470,127,5/51,19.44,8.53,13.6,2,1,Arshdeep Singh,IND
2,Ali Dawood (BHR),75,75,266.4,8,1656,116,7/19,14.27,6.21,13.7,6,1,Ali Dawood,BHR
3,Ehsan Khan (HKG),79,77,283.1,5,1771,113,4/5,15.67,6.25,15,5,0,Ehsan Khan,HKG
4,Junaid Siddique (UAE),77,77,290.1,2,2273,108,5/35,21.04,7.83,16.1,4,1,Junaid Siddique,UAE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3322,Zubaidi Zulkifle (MAS),65,0,0,0,0,0,0,0,0,0,0,0,Zubaidi Zulkifle,MAS
3323,Zubair Ali (QAT),14,0,0,0,0,0,0,0,0,0,0,0,Zubair Ali,QAT
3324,Zuhair Muhammad (KSA),3,0,0,0,0,0,0,0,0,0,0,0,Zuhair Muhammad,KSA
3325,L Zulu (SWZ),4,0,0,0,0,0,0,0,0,0,0,0,L Zulu,SWZ


In [26]:
df4.to_csv('data/Overall_bowlers.csv')


# Positionwise Batting Stats

## Upper Order

In [27]:
HEADERS = {'User-Agent': 'CricinfoDataProject/1.0 (contact: arkosaha.ruet@gmail.com)'}
# Define the base URL and the initial page URL
base_url = 'https://stats.espncricinfo.com'
url = base_url + '/ci/engine/stats/index.html?batting_positionmax2=3;batting_positionmin2=1;batting_positionval2=batting_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=runs;page=1;size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=batting'

response = requests.get(url, headers=HEADERS)
response = response.content
soup = BeautifulSoup(response, 'html.parser')


In [28]:
# Initialize the page number
page_number = 1
total_pages = int(soup.find_all('td', class_='left')[6].text.split(' ')[6].rstrip())


In [29]:
page_number


1

In [30]:
total_pages


8

In [31]:
data = []
for i in range(1, total_pages+1):
  url = f"https://stats.espncricinfo.com/ci/engine/stats/index.html?batting_positionmax2=3;batting_positionmin2=1;batting_positionval2=batting_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=runs;page={i};size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=batting"
  time.sleep(1.5)
  response = requests.get(url, headers=HEADERS)
  response = response.content
  soup = BeautifulSoup(response, 'html.parser')
  # Find the table element that contains the data we want to extract
  table = soup.select_one('#ciHomeContentlhs > div.pnl650M > table:nth-child(5)')

  # Check if the table exists
  if table is not None:
    # Loop through the rows of the table and extract the data for each player
    for row in table.tbody.find_all('tr'):
      columns = row.find_all('td')
      if len(columns) > 0:
        Player = columns[0].text.strip()
        Span = columns[1].text.strip()
        Matches = columns[2].text.strip()
        Innings = columns[3].text.strip()
        Not_Out = columns[4].text.strip()
        Runs = columns[5].text.strip()
        Highest_Score = columns[6].text.strip()
        Average = columns[7].text.strip()
        Balls_Faced = columns[8].text.strip()
        SR = columns[9].text.strip()
        Hundreds = columns[10].text.strip()
        Fifties = columns[11].text.strip()
        Zeros = columns[12].text.strip()
        Fours = columns[13].text.strip()
        Sixes = columns[14].text.strip()


        # Add the data for each player to the DataFrame
        data.append({
          'Player': Player,
          'Span': Span,
          'Matches': Matches,
          'Innings': Innings,
          'Not_Out' : Not_Out,
          'Runs': Runs,
          'Highest_Score': Highest_Score,
          'Average' : Average,
          'Balls_Faced': Balls_Faced,
          'SR': SR,
          'Hundreds': Hundreds,
          'Fifties': Fifties,
          'Zeros' : Zeros,
          'Fours': Fours,
          'Sixes': Sixes
        })

    # Check if there is a next page by finding the 'Next' button on the page
    next_page_link = soup.select_one('.PaginationLink')
    if next_page_link is not None:
      # If there is a next page, increment the page number and update the URL to the next page
      i += 1
    else:
      # If there is no next page, break out of the loop
      break

# Create a DataFrame from the scraped data
df5 = pd.DataFrame(data)


In [32]:
# Print the DataFrame
df5.head(10)


,Player,Span,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,Muhammad Waseem (UAE),2022-2026,92,92,5,3190,112,36.66,2109,151.25,2,26,5,262,187
1,P Nissanka (SL),2022-2026,79,79,5,2346,107,31.7,1800,130.33,2,16,7,243,69
2,BKG Mendis (SL),2022-2026,73,73,6,2037,86,30.4,1539,132.35,0,15,5,189,73
3,JC Buttler (ENG),2022-2026,60,60,4,1840,96,32.85,1181,155.8,0,13,6,186,81
4,BJ Bennett (ZIM),2023-2026,54,54,4,1833,111,36.66,1272,144.1,1,12,2,212,55
5,Babar Azam (PAK),2022-2025,61,61,7,1809,110*,33.5,1419,127.48,2,13,7,194,37
6,Ibrahim Zadran (AFG),2022-2026,59,59,8,1781,95*,34.92,1540,115.64,0,16,1,167,41
7,Mohammad Rizwan (PAK),2022-2024,49,49,9,1775,98*,44.37,1452,122.24,0,17,2,135,48
8,Litton Das (BAN),2022-2025,72,71,4,1760,83,26.26,1371,128.37,0,12,2,168,54
9,Rahmanullah Gurbaz (AFG),2022-2026,69,69,0,1749,100,25.34,1301,134.43,1,10,8,141,87


In [33]:
df5 = df5.drop('Span', axis=1)
df5


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,Muhammad Waseem (UAE),92,92,5,3190,112,36.66,2109,151.25,2,26,5,262,187
1,P Nissanka (SL),79,79,5,2346,107,31.7,1800,130.33,2,16,7,243,69
2,BKG Mendis (SL),73,73,6,2037,86,30.4,1539,132.35,0,15,5,189,73
3,JC Buttler (ENG),60,60,4,1840,96,32.85,1181,155.8,0,13,6,186,81
4,BJ Bennett (ZIM),54,54,4,1833,111,36.66,1272,144.1,1,12,2,212,55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1432,B Pondani (MWI),1,-,-,-,-,-,-,-,-,-,-,-,-
1433,CR Richards (STHEL),1,-,-,-,-,-,-,-,-,-,-,-,-
1434,Robert Raina (THA),1,-,-,-,-,-,-,-,-,-,-,-,-
1435,O Sam Arthur (CRC),1,-,-,-,-,-,-,-,-,-,-,-,-


In [34]:
def replace_values(value):
    if value == 'DNB':
      return 0
    elif value == 'TDNB':
      return 0
    elif value == 'absent':
      return 0
    elif value == 'sub':
      return 0
    elif value == '-':
      return 0
    else:
      return value

# apply the replace_values function to all columns in the DataFrame
df5 = df5.map(replace_values)
df5


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,Muhammad Waseem (UAE),92,92,5,3190,112,36.66,2109,151.25,2,26,5,262,187
1,P Nissanka (SL),79,79,5,2346,107,31.7,1800,130.33,2,16,7,243,69
2,BKG Mendis (SL),73,73,6,2037,86,30.4,1539,132.35,0,15,5,189,73
3,JC Buttler (ENG),60,60,4,1840,96,32.85,1181,155.8,0,13,6,186,81
4,BJ Bennett (ZIM),54,54,4,1833,111,36.66,1272,144.1,1,12,2,212,55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1432,B Pondani (MWI),1,0,0,0,0,0,0,0,0,0,0,0,0
1433,CR Richards (STHEL),1,0,0,0,0,0,0,0,0,0,0,0,0
1434,Robert Raina (THA),1,0,0,0,0,0,0,0,0,0,0,0,0
1435,O Sam Arthur (CRC),1,0,0,0,0,0,0,0,0,0,0,0,0


In [35]:
# Extract player name and country using regular expression
df5[['Player_Name', 'Country']] = df5['Player'].str.extract(r'^(.*?) \((.*?)\)')


In [36]:
df5


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes,Player_Name,Country
0,Muhammad Waseem (UAE),92,92,5,3190,112,36.66,2109,151.25,2,26,5,262,187,Muhammad Waseem,UAE
1,P Nissanka (SL),79,79,5,2346,107,31.7,1800,130.33,2,16,7,243,69,P Nissanka,SL
2,BKG Mendis (SL),73,73,6,2037,86,30.4,1539,132.35,0,15,5,189,73,BKG Mendis,SL
3,JC Buttler (ENG),60,60,4,1840,96,32.85,1181,155.8,0,13,6,186,81,JC Buttler,ENG
4,BJ Bennett (ZIM),54,54,4,1833,111,36.66,1272,144.1,1,12,2,212,55,BJ Bennett,ZIM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1432,B Pondani (MWI),1,0,0,0,0,0,0,0,0,0,0,0,0,B Pondani,MWI
1433,CR Richards (STHEL),1,0,0,0,0,0,0,0,0,0,0,0,0,CR Richards,STHEL
1434,Robert Raina (THA),1,0,0,0,0,0,0,0,0,0,0,0,0,Robert Raina,THA
1435,O Sam Arthur (CRC),1,0,0,0,0,0,0,0,0,0,0,0,0,O Sam Arthur,CRC


In [37]:
df5.to_csv('data/Upper_Order.csv')


## Middle-Order

In [38]:
HEADERS = {'User-Agent': 'CricinfoDataProject/1.0 (contact: arkosaha.ruet@gmail.com)'}
# Define the base URL and the initial page URL
base_url = 'https://stats.espncricinfo.com'
url = base_url + '/ci/engine/stats/index.html?batting_positionmax2=7;batting_positionmin2=4;batting_positionval2=batting_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=runs;page=1;size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=batting'

response = requests.get(url, headers=HEADERS)
response = response.content
soup = BeautifulSoup(response, 'html.parser')


In [39]:
# Initialize the page number
page_number = 1
total_pages = int(soup.find_all('td', class_='left')[6].text.split(' ')[6].rstrip())


In [40]:
page_number


1

In [41]:
total_pages


11

In [42]:
data = []
for i in range(1, total_pages+1):
  url = f"https://stats.espncricinfo.com/ci/engine/stats/index.html?batting_positionmax2=7;batting_positionmin2=4;batting_positionval2=batting_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=runs;page={i};size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=batting"
  time.sleep(1.5)
  response = requests.get(url, headers=HEADERS)
  response = response.content
  soup = BeautifulSoup(response, 'html.parser')
  # Find the table element that contains the data we want to extract
  table = soup.select_one('#ciHomeContentlhs > div.pnl650M > table:nth-child(5)')

  # Check if the table exists
  if table is not None:
    # Loop through the rows of the table and extract the data for each player
    for row in table.tbody.find_all('tr'):
      columns = row.find_all('td')
      if len(columns) > 0:
        Player = columns[0].text.strip()
        Span = columns[1].text.strip()
        Matches = columns[2].text.strip()
        Innings = columns[3].text.strip()
        Not_Out = columns[4].text.strip()
        Runs = columns[5].text.strip()
        Highest_Score = columns[6].text.strip()
        Average = columns[7].text.strip()
        Balls_Faced = columns[8].text.strip()
        SR = columns[9].text.strip()
        Hundreds = columns[10].text.strip()
        Fifties = columns[11].text.strip()
        Zeros = columns[12].text.strip()
        Fours = columns[13].text.strip()
        Sixes = columns[14].text.strip()


        # Add the data for each player to the DataFrame
        data.append({
          'Player': Player,
          'Span': Span,
          'Matches': Matches,
          'Innings': Innings,
          'Not_Out' : Not_Out,
          'Runs': Runs,
          'Highest_Score': Highest_Score,
          'Average' : Average,
          'Balls_Faced': Balls_Faced,
          'SR': SR,
          'Hundreds': Hundreds,
          'Fifties': Fifties,
          'Zeros' : Zeros,
          'Fours': Fours,
          'Sixes': Sixes
        })

    # Check if there is a next page by finding the 'Next' button on the page
    next_page_link = soup.select_one('.PaginationLink')
    if next_page_link is not None:
      # If there is a next page, increment the page number and update the URL to the next page
      i += 1
    else:
      # If there is no next page, break out of the loop
      break

# Create a DataFrame from the scraped data
df6 = pd.DataFrame(data)


In [43]:
# Print the DataFrame
df6.head(10)


,Player,Span,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,Sikandar Raza (ZIM),2022-2026,83,80,10,2242,133*,32.02,1561,143.62,1,12,6,170,112
1,SA Yadav (IND),2022-2026,68,66,11,2205,117,40.09,1336,165.04,3,17,2,192,122
2,R Powell (WI),2022-2026,82,77,12,1882,107,28.95,1274,147.72,1,8,0,106,134
3,HH Pandya (IND),2022-2026,83,72,19,1717,71*,32.39,1177,145.87,0,9,2,133,93
4,GD Phillips (NZ),2022-2026,59,53,8,1613,104,35.84,1121,143.88,1,10,1,126,70
5,DS Airee (NEP),2022-2026,57,54,12,1464,110*,34.85,1027,142.55,1,10,1,100,64
6,DJ Mitchell (NZ),2022-2026,70,62,14,1423,72*,29.64,984,144.61,0,7,1,94,56
7,Haider Butt (BHR),2022-2024,56,54,13,1384,79*,33.75,1134,122.04,0,8,1,88,62
8,Sohail Ahmed (BHR),2022-2026,55,53,18,1382,80*,39.48,1104,125.18,0,11,4,89,64
9,Ahmer Bin (BHR),2022-2026,79,66,20,1337,68*,29.06,1044,128.06,0,6,1,103,45


In [44]:
df6 = df6.drop('Span', axis=1)
df6


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,Sikandar Raza (ZIM),83,80,10,2242,133*,32.02,1561,143.62,1,12,6,170,112
1,SA Yadav (IND),68,66,11,2205,117,40.09,1336,165.04,3,17,2,192,122
2,R Powell (WI),82,77,12,1882,107,28.95,1274,147.72,1,8,0,106,134
3,HH Pandya (IND),83,72,19,1717,71*,32.39,1177,145.87,0,9,2,133,93
4,GD Phillips (NZ),59,53,8,1613,104,35.84,1121,143.88,1,10,1,126,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2169,SMK Waththage (CZK-R),1,-,-,-,-,-,-,-,-,-,-,-,-
2170,CG Williams (NAM),1,-,-,-,-,-,-,-,-,-,-,-,-
2171,A Wright (CAY),1,-,-,-,-,-,-,-,-,-,-,-,-
2172,HN Ya France (NAM),2,-,-,-,-,-,-,-,-,-,-,-,-


In [45]:
def replace_values(value):
    if value == 'DNB':
      return 0
    elif value == 'TDNB':
      return 0
    elif value == 'absent':
      return 0
    elif value == 'sub':
      return 0
    elif value == '-':
      return 0
    else:
      return value

# apply the replace_values function to all columns in the DataFrame
df6 = df6.map(replace_values)
df6


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,Sikandar Raza (ZIM),83,80,10,2242,133*,32.02,1561,143.62,1,12,6,170,112
1,SA Yadav (IND),68,66,11,2205,117,40.09,1336,165.04,3,17,2,192,122
2,R Powell (WI),82,77,12,1882,107,28.95,1274,147.72,1,8,0,106,134
3,HH Pandya (IND),83,72,19,1717,71*,32.39,1177,145.87,0,9,2,133,93
4,GD Phillips (NZ),59,53,8,1613,104,35.84,1121,143.88,1,10,1,126,70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2169,SMK Waththage (CZK-R),1,0,0,0,0,0,0,0,0,0,0,0,0
2170,CG Williams (NAM),1,0,0,0,0,0,0,0,0,0,0,0,0
2171,A Wright (CAY),1,0,0,0,0,0,0,0,0,0,0,0,0
2172,HN Ya France (NAM),2,0,0,0,0,0,0,0,0,0,0,0,0


In [46]:
# Extract player name and country using regular expression
df6[['Player_Name', 'Country']] = df6['Player'].str.extract(r'^(.*?) \((.*?)\)')


In [47]:
df6


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes,Player_Name,Country
0,Sikandar Raza (ZIM),83,80,10,2242,133*,32.02,1561,143.62,1,12,6,170,112,Sikandar Raza,ZIM
1,SA Yadav (IND),68,66,11,2205,117,40.09,1336,165.04,3,17,2,192,122,SA Yadav,IND
2,R Powell (WI),82,77,12,1882,107,28.95,1274,147.72,1,8,0,106,134,R Powell,WI
3,HH Pandya (IND),83,72,19,1717,71*,32.39,1177,145.87,0,9,2,133,93,HH Pandya,IND
4,GD Phillips (NZ),59,53,8,1613,104,35.84,1121,143.88,1,10,1,126,70,GD Phillips,NZ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2169,SMK Waththage (CZK-R),1,0,0,0,0,0,0,0,0,0,0,0,0,SMK Waththage,CZK-R
2170,CG Williams (NAM),1,0,0,0,0,0,0,0,0,0,0,0,0,CG Williams,NAM
2171,A Wright (CAY),1,0,0,0,0,0,0,0,0,0,0,0,0,A Wright,CAY
2172,HN Ya France (NAM),2,0,0,0,0,0,0,0,0,0,0,0,0,HN Ya France,NAM


In [48]:
df6.to_csv('data/Middle_Order.csv')


## Lower Order

In [49]:
HEADERS = {'User-Agent': 'CricinfoDataProject/1.0 (contact: arkosaha.ruet@gmail.com)'}
# Define the base URL and the initial page URL
base_url = 'https://stats.espncricinfo.com'
url = base_url + '/ci/engine/stats/index.html?batting_positionmax2=11;batting_positionmin2=8;batting_positionval2=batting_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=runs;page=1;size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=batting'

response = requests.get(url, headers=HEADERS)
response = response.content
soup = BeautifulSoup(response, 'html.parser')


In [50]:
# Initialize the page number
page_number = 1
total_pages = int(soup.find_all('td', class_='left')[6].text.split(' ')[6].rstrip())


In [51]:
page_number


1

In [52]:
total_pages


12

In [53]:
data = []
for i in range(1, total_pages+1):
  url = f"https://stats.espncricinfo.com/ci/engine/stats/index.html?batting_positionmax2=11;batting_positionmin2=8;batting_positionval2=batting_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=runs;page={i};size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=batting"
  time.sleep(1.5)
  response = requests.get(url, headers=HEADERS)
  response = response.content
  soup = BeautifulSoup(response, 'html.parser')
  # Find the table element that contains the data we want to extract
  table = soup.select_one('#ciHomeContentlhs > div.pnl650M > table:nth-child(5)')

  # Check if the table exists
  if table is not None:
    # Loop through the rows of the table and extract the data for each player
    for row in table.tbody.find_all('tr'):
      columns = row.find_all('td')
      if len(columns) > 0:
        Player = columns[0].text.strip()
        Span = columns[1].text.strip()
        Matches = columns[2].text.strip()
        Innings = columns[3].text.strip()
        Not_Out = columns[4].text.strip()
        Runs = columns[5].text.strip()
        Highest_Score = columns[6].text.strip()
        Average = columns[7].text.strip()
        Balls_Faced = columns[8].text.strip()
        SR = columns[9].text.strip()
        Hundreds = columns[10].text.strip()
        Fifties = columns[11].text.strip()
        Zeros = columns[12].text.strip()
        Fours = columns[13].text.strip()
        Sixes = columns[14].text.strip()


        # Add the data for each player to the DataFrame
        data.append({
          'Player': Player,
          'Span': Span,
          'Matches': Matches,
          'Innings': Innings,
          'Not_Out' : Not_Out,
          'Runs': Runs,
          'Highest_Score': Highest_Score,
          'Average' : Average,
          'Balls_Faced': Balls_Faced,
          'SR': SR,
          'Hundreds': Hundreds,
          'Fifties': Fifties,
          'Zeros' : Zeros,
          'Fours': Fours,
          'Sixes': Sixes
        })

    # Check if there is a next page by finding the 'Next' button on the page
    next_page_link = soup.select_one('.PaginationLink')
    if next_page_link is not None:
      # If there is a next page, increment the page number and update the URL to the next page
      i += 1
    else:
      # If there is no next page, break out of the loop
      break

# Create a DataFrame from the scraped data
df7 = pd.DataFrame(data)


In [54]:
# Print the DataFrame
df7.head(10)


,Player,Span,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,MR Adair (IRE),2022-2026,56,36,6,498,72,16.6,370,134.59,0,1,3,39,23
1,R Shepherd (WI),2022-2026,33,22,8,453,52*,32.35,316,143.35,0,1,0,29,31
2,Rashid Khan (AFG),2022-2026,43,30,9,348,48*,16.57,264,131.81,0,0,1,27,21
3,Shaheen Shah Afridi (PAK),2022-2026,58,34,15,292,33*,15.36,229,127.51,0,0,6,16,22
4,Faheem Ashraf (PAK),2023-2026,34,23,5,284,51,15.77,202,140.59,0,1,2,24,14
5,LM Jongwe (ZIM),2022-2024,35,25,7,279,35,15.5,228,122.36,0,0,2,27,7
6,Karan KC (NEP),2022-2026,48,28,12,250,33*,15.62,166,150.6,0,0,2,13,19
7,Shakeel Ahmed (OMA),2023-2026,46,25,7,243,45,13.5,253,96.04,0,0,3,16,9
8,Z Bimenyimana (RWN),2022-2025,80,51,16,233,25*,6.65,197,118.27,0,0,13,13,13
9,BJ McCarthy (IRE),2022-2026,52,31,12,225,51*,11.84,193,116.58,0,1,3,14,13


In [55]:
df7 = df7.drop('Span', axis=1)
df7


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,MR Adair (IRE),56,36,6,498,72,16.6,370,134.59,0,1,3,39,23
1,R Shepherd (WI),33,22,8,453,52*,32.35,316,143.35,0,1,0,29,31
2,Rashid Khan (AFG),43,30,9,348,48*,16.57,264,131.81,0,0,1,27,21
3,Shaheen Shah Afridi (PAK),58,34,15,292,33*,15.36,229,127.51,0,0,6,16,22
4,Faheem Ashraf (PAK),34,23,5,284,51,15.77,202,140.59,0,1,2,24,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2368,Zeeshan Maqsood (OMA),3,-,-,-,-,-,-,-,-,-,-,-,-
2369,Ziaur Rahman (AFG),5,-,-,-,-,-,-,-,-,-,-,-,-
2370,Zikria Islam (OMA),1,-,-,-,-,-,-,-,-,-,-,-,-
2371,F Zirahangaje (RWN),2,-,-,-,-,-,-,-,-,-,-,-,-


In [56]:
def replace_values(value):
    if value == 'DNB':
      return 0
    elif value == 'TDNB':
      return 0
    elif value == 'absent':
      return 0
    elif value == 'sub':
      return 0
    elif value == '-':
      return 0
    else:
      return value

# apply the replace_values function to all columns in the DataFrame
df7 = df7.map(replace_values)
df7


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes
0,MR Adair (IRE),56,36,6,498,72,16.6,370,134.59,0,1,3,39,23
1,R Shepherd (WI),33,22,8,453,52*,32.35,316,143.35,0,1,0,29,31
2,Rashid Khan (AFG),43,30,9,348,48*,16.57,264,131.81,0,0,1,27,21
3,Shaheen Shah Afridi (PAK),58,34,15,292,33*,15.36,229,127.51,0,0,6,16,22
4,Faheem Ashraf (PAK),34,23,5,284,51,15.77,202,140.59,0,1,2,24,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2368,Zeeshan Maqsood (OMA),3,0,0,0,0,0,0,0,0,0,0,0,0
2369,Ziaur Rahman (AFG),5,0,0,0,0,0,0,0,0,0,0,0,0
2370,Zikria Islam (OMA),1,0,0,0,0,0,0,0,0,0,0,0,0
2371,F Zirahangaje (RWN),2,0,0,0,0,0,0,0,0,0,0,0,0


In [57]:
# Extract player name and country using regular expression
df7[['Player_Name', 'Country']] = df7['Player'].str.extract(r'^(.*?) \((.*?)\)')


In [58]:
df7


,Player,Matches,Innings,Not_Out,Runs,Highest_Score,Average,Balls_Faced,SR,Hundreds,Fifties,Zeros,Fours,Sixes,Player_Name,Country
0,MR Adair (IRE),56,36,6,498,72,16.6,370,134.59,0,1,3,39,23,MR Adair,IRE
1,R Shepherd (WI),33,22,8,453,52*,32.35,316,143.35,0,1,0,29,31,R Shepherd,WI
2,Rashid Khan (AFG),43,30,9,348,48*,16.57,264,131.81,0,0,1,27,21,Rashid Khan,AFG
3,Shaheen Shah Afridi (PAK),58,34,15,292,33*,15.36,229,127.51,0,0,6,16,22,Shaheen Shah Afridi,PAK
4,Faheem Ashraf (PAK),34,23,5,284,51,15.77,202,140.59,0,1,2,24,14,Faheem Ashraf,PAK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2368,Zeeshan Maqsood (OMA),3,0,0,0,0,0,0,0,0,0,0,0,0,Zeeshan Maqsood,OMA
2369,Ziaur Rahman (AFG),5,0,0,0,0,0,0,0,0,0,0,0,0,Ziaur Rahman,AFG
2370,Zikria Islam (OMA),1,0,0,0,0,0,0,0,0,0,0,0,0,Zikria Islam,OMA
2371,F Zirahangaje (RWN),2,0,0,0,0,0,0,0,0,0,0,0,0,F Zirahangaje,RWN


In [59]:
df7.to_csv('data/Lower_Order.csv')


# Positionwise Bowling Stats

## Opening

In [60]:
HEADERS = {'User-Agent': 'CricinfoDataProject/1.0 (contact: arkosaha.ruet@gmail.com)'}
# Define the initial page URL
url = 'https://stats.espncricinfo.com/ci/engine/stats/index.html?bowling_positionmax1=2;bowling_positionmax2=2;bowling_positionmin1=1;bowling_positionmin2=1;bowling_positionval1=bowling_position;bowling_positionval2=bowling_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=wickets;page=1;size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling'

response = requests.get(url, headers=HEADERS)
response = response.content
soup = BeautifulSoup(response, 'html.parser')


In [61]:
# Initialize the page number
page_number = 1
total_pages = int(soup.find_all('td', class_='left')[6].text.split(' ')[6].rstrip())


In [62]:
page_number


1

In [63]:
total_pages


6

In [64]:
data = []
for i in range(1, total_pages+1):
  url = f"https://stats.espncricinfo.com/ci/engine/stats/index.html?bowling_positionmax1=2;bowling_positionmax2=2;bowling_positionmin1=1;bowling_positionmin2=1;bowling_positionval1=bowling_position;bowling_positionval2=bowling_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=wickets;page={i};size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling"
  time.sleep(1.5)
  response = requests.get(url, headers=HEADERS)
  response = response.content
  soup = BeautifulSoup(response, 'html.parser')
  # Find the table element that contains the data we want to extract
  table = soup.select_one('#ciHomeContentlhs > div.pnl650M > table:nth-child(5)')

  # Check if the table exists
  if table is not None:
    # Loop through the rows of the table and extract the data for each player
    for row in table.tbody.find_all('tr'):
      columns = row.find_all('td')
      if len(columns) > 0:
        Player = columns[0].text.strip()
        Span = columns[1].text.strip()
        Matches = columns[2].text.strip()
        Innings = columns[3].text.strip()
        Overs = columns[4].text.strip()
        Maidens = columns[5].text.strip()
        Runs = columns[6].text.strip()
        Wickets = columns[7].text.strip()
        BBI = columns[8].text.strip()
        Average = columns[9].text.strip()
        Economy = columns[10].text.strip()
        SR = columns[11].text.strip()
        Four_Wickets = columns[12].text.strip()
        Five_Wickets = columns[13].text.strip()


        # Add the data for each player to the DataFrame
        data.append({
          'Player': Player,
          'Span': Span,
          'Matches': Matches,
          'Innings': Innings,
          'Overs' : Overs,
          'Maidens': Maidens,
          'Runs': Runs,
          'Wickets' : Wickets,
          'BBI': BBI,
          'Average': Average,
          'Economy': Economy,
          'SR': SR,
          'Four_Wickets' : Four_Wickets,
          'Five_Wickets': Five_Wickets
        })

    # Check if there is a next page by finding the 'Next' button on the page
    next_page_link = soup.select_one('.PaginationLink')
    if next_page_link is not None:
      # If there is a next page, increment the page number and update the URL to the next page
      i += 1
    else:
      # If there is no next page, break out of the loop
      break

# Create a DataFrame from the scraped data
df8 = pd.DataFrame(data)


In [65]:
# Print the DataFrame
df8.head(10)


,Player,Span,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets
0,Arshdeep Singh (IND),2022-2026,77,77,276.1,2,2398,119,5/51,20.15,8.68,13.9,2,1
1,Ali Dawood (BHR),2023-2026,75,75,266.4,8,1656,116,7/19,14.27,6.21,13.7,6,1
2,Junaid Siddique (UAE),2022-2026,69,69,262.4,2,2029,95,5/35,21.35,7.72,16.5,4,1
3,MR Adair (IRE),2022-2026,69,69,246.0,1,2014,91,4/13,22.13,8.18,16.2,2,0
4,Shaheen Shah Afridi (PAK),2022-2026,64,64,227.1,1,1782,91,4/22,19.58,7.84,14.9,3,0
5,R Ngarava (ZIM),2022-2026,68,68,247.2,8,1800,90,4/16,20,7.27,16.4,2,0
6,Z Bimenyimana (RWN),2022-2025,84,84,255.0,7,1809,86,3/7,21.03,7.09,17.7,0,0
7,Rizwan Butt (BHR),2022-2026,53,53,191.5,4,1307,84,5/12,15.55,6.81,13.7,2,3
8,Pavandeep Singh (MAS),2022-2025,60,60,226.4,7,1283,71,4/16,18.07,5.66,19.1,1,0
9,Fazalhaq Farooqi (AFG),2022-2026,53,53,185.3,4,1304,65,5/9,20.06,7.02,17.1,1,1


In [66]:
# Extract player name and country using regular expression
df8[['Player_Name', 'Country']] = df8['Player'].str.extract(r'^(.*?) \((.*?)\)')


In [67]:
df8


,Player,Span,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Arshdeep Singh (IND),2022-2026,77,77,276.1,2,2398,119,5/51,20.15,8.68,13.9,2,1,Arshdeep Singh,IND
1,Ali Dawood (BHR),2023-2026,75,75,266.4,8,1656,116,7/19,14.27,6.21,13.7,6,1,Ali Dawood,BHR
2,Junaid Siddique (UAE),2022-2026,69,69,262.4,2,2029,95,5/35,21.35,7.72,16.5,4,1,Junaid Siddique,UAE
3,MR Adair (IRE),2022-2026,69,69,246.0,1,2014,91,4/13,22.13,8.18,16.2,2,0,MR Adair,IRE
4,Shaheen Shah Afridi (PAK),2022-2026,64,64,227.1,1,1782,91,4/22,19.58,7.84,14.9,3,0,Shaheen Shah Afridi,PAK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1175,JJ Wright (SWZ),2024-2024,1,1,1.4,0,14,0,-,-,8.4,-,0,0,JJ Wright,SWZ
1176,J Yon (STHEL),2024-2024,1,1,1.0,0,12,0,-,-,12,-,0,0,J Yon,STHEL
1177,Zafer Durmaz (TKY),2024-2024,1,1,2.0,0,16,0,-,-,8,-,0,0,Zafer Durmaz,TKY
1178,Zahid Ali (UAE),2025-2025,1,1,2.0,0,26,0,-,-,13,-,0,0,Zahid Ali,UAE


In [68]:
df8 = df8.drop('Span', axis=1)
df8


,Player,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Arshdeep Singh (IND),77,77,276.1,2,2398,119,5/51,20.15,8.68,13.9,2,1,Arshdeep Singh,IND
1,Ali Dawood (BHR),75,75,266.4,8,1656,116,7/19,14.27,6.21,13.7,6,1,Ali Dawood,BHR
2,Junaid Siddique (UAE),69,69,262.4,2,2029,95,5/35,21.35,7.72,16.5,4,1,Junaid Siddique,UAE
3,MR Adair (IRE),69,69,246.0,1,2014,91,4/13,22.13,8.18,16.2,2,0,MR Adair,IRE
4,Shaheen Shah Afridi (PAK),64,64,227.1,1,1782,91,4/22,19.58,7.84,14.9,3,0,Shaheen Shah Afridi,PAK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1175,JJ Wright (SWZ),1,1,1.4,0,14,0,-,-,8.4,-,0,0,JJ Wright,SWZ
1176,J Yon (STHEL),1,1,1.0,0,12,0,-,-,12,-,0,0,J Yon,STHEL
1177,Zafer Durmaz (TKY),1,1,2.0,0,16,0,-,-,8,-,0,0,Zafer Durmaz,TKY
1178,Zahid Ali (UAE),1,1,2.0,0,26,0,-,-,13,-,0,0,Zahid Ali,UAE


In [69]:
def replace_values(value):
    if value == 'DNB':
      return 0
    elif value == 'TDNB':
      return 0
    elif value == 'absent':
      return 0
    elif value == 'sub':
      return 0
    elif value == '-':
      return 0
    else:
      return value

# apply the replace_values function to all columns in the DataFrame
df8 = df8.map(replace_values)
df8


,Player,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Arshdeep Singh (IND),77,77,276.1,2,2398,119,5/51,20.15,8.68,13.9,2,1,Arshdeep Singh,IND
1,Ali Dawood (BHR),75,75,266.4,8,1656,116,7/19,14.27,6.21,13.7,6,1,Ali Dawood,BHR
2,Junaid Siddique (UAE),69,69,262.4,2,2029,95,5/35,21.35,7.72,16.5,4,1,Junaid Siddique,UAE
3,MR Adair (IRE),69,69,246.0,1,2014,91,4/13,22.13,8.18,16.2,2,0,MR Adair,IRE
4,Shaheen Shah Afridi (PAK),64,64,227.1,1,1782,91,4/22,19.58,7.84,14.9,3,0,Shaheen Shah Afridi,PAK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1175,JJ Wright (SWZ),1,1,1.4,0,14,0,0,0,8.4,0,0,0,JJ Wright,SWZ
1176,J Yon (STHEL),1,1,1.0,0,12,0,0,0,12,0,0,0,J Yon,STHEL
1177,Zafer Durmaz (TKY),1,1,2.0,0,16,0,0,0,8,0,0,0,Zafer Durmaz,TKY
1178,Zahid Ali (UAE),1,1,2.0,0,26,0,0,0,13,0,0,0,Zahid Ali,UAE


In [70]:
df8.to_csv('data/Opening_Bowlers.csv')


## First Change

In [71]:
HEADERS = {'User-Agent': 'CricinfoDataProject/1.0 (contact: arkosaha.ruet@gmail.com)'}
# Define the initial page URL
url = 'https://stats.espncricinfo.com/ci/engine/stats/index.html?bowling_positionmax2=3;bowling_positionmin2=3;bowling_positionval2=bowling_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=wickets;page=1;size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling'

response = requests.get(url, headers=HEADERS)
response = response.content
soup = BeautifulSoup(response, 'html.parser')


In [72]:
# Initialize the page number
page_number = 1
total_pages = int(soup.find_all('td', class_='left')[6].text.split(' ')[6].rstrip())


In [73]:
page_number


1

In [74]:
total_pages


6

In [75]:
data = []
for i in range(1, total_pages+1):
  url = f"https://stats.espncricinfo.com/ci/engine/stats/index.html?bowling_positionmax2=3;bowling_positionmin2=3;bowling_positionval2=bowling_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=wickets;page={i};size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling"
  time.sleep(1.5)
  response = requests.get(url, headers=HEADERS)
  response = response.content
  soup = BeautifulSoup(response, 'html.parser')
  # Find the table element that contains the data we want to extract
  table = soup.select_one('#ciHomeContentlhs > div.pnl650M > table:nth-child(5)')

  # Check if the table exists
  if table is not None:
    # Loop through the rows of the table and extract the data for each player
    for row in table.tbody.find_all('tr'):
      columns = row.find_all('td')
      if len(columns) > 0:
        Player = columns[0].text.strip()
        Span = columns[1].text.strip()
        Matches = columns[2].text.strip()
        Innings = columns[3].text.strip()
        Overs = columns[4].text.strip()
        Maidens = columns[5].text.strip()
        Runs = columns[6].text.strip()
        Wickets = columns[7].text.strip()
        BBI = columns[8].text.strip()
        Average = columns[9].text.strip()
        Economy = columns[10].text.strip()
        SR = columns[11].text.strip()
        Four_Wickets = columns[12].text.strip()
        Five_Wickets = columns[13].text.strip()


        # Add the data for each player to the DataFrame
        data.append({
          'Player': Player,
          'Span': Span,
          'Matches': Matches,
          'Innings': Innings,
          'Overs' : Overs,
          'Maidens': Maidens,
          'Runs': Runs,
          'Wickets' : Wickets,
          'BBI': BBI,
          'Average': Average,
          'Economy': Economy,
          'SR': SR,
          'Four_Wickets' : Four_Wickets,
          'Five_Wickets': Five_Wickets
        })

    # Check if there is a next page by finding the 'Next' button on the page
    next_page_link = soup.select_one('.PaginationLink')
    if next_page_link is not None:
      # If there is a next page, increment the page number and update the URL to the next page
      i += 1
    else:
      # If there is no next page, break out of the loop
      break

# Create a DataFrame from the scraped data
df9 = pd.DataFrame(data)


In [76]:
# Print the DataFrame
df9.head(10)


,Player,Span,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets
0,AM Kimote (TAN),2022-2025,33,33,102.5,3,691,48,4/21,14.39,6.71,12.8,1,0
1,BJ McCarthy (IRE),2022-2026,37,37,133.0,2,1103,45,3/7,24.51,8.29,17.7,0,0
2,Ehsan Khan (HKG),2022-2025,31,31,114.5,3,731,44,4/28,16.61,6.36,15.6,1,0
3,B Evans (ZIM),2022-2026,23,23,76.4,0,583,39,4/16,14.94,7.6,11.7,1,0
4,N Senamontree (THA),2023-2026,26,26,98.5,10,486,39,4/9,12.46,4.91,15.2,2,0
5,Haris Rauf (PAK),2022-2025,29,29,103.0,0,941,37,4/22,25.43,9.13,16.7,1,0
6,M Akayezu (RWN),2022-2025,34,34,102.2,2,773,36,3/6,21.47,7.55,17,0,0
7,Khalid Ahmadi (BEL),2022-2025,19,19,70.1,1,470,35,4/5,13.42,6.69,12,2,0
8,Rizwan Butt (BHR),2023-2026,23,23,81.5,4,501,33,6/9,15.18,6.12,14.8,0,1
9,CA Young (IRE),2022-2025,18,18,64.5,0,538,32,4/28,16.81,8.29,12.1,1,0


In [77]:
# Extract player name and country using regular expression
df9[['Player_Name', 'Country']] = df9['Player'].str.extract(r'^(.*?) \((.*?)\)')


In [78]:
df9


,Player,Span,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,AM Kimote (TAN),2022-2025,33,33,102.5,3,691,48,4/21,14.39,6.71,12.8,1,0,AM Kimote,TAN
1,BJ McCarthy (IRE),2022-2026,37,37,133.0,2,1103,45,3/7,24.51,8.29,17.7,0,0,BJ McCarthy,IRE
2,Ehsan Khan (HKG),2022-2025,31,31,114.5,3,731,44,4/28,16.61,6.36,15.6,1,0,Ehsan Khan,HKG
3,B Evans (ZIM),2022-2026,23,23,76.4,0,583,39,4/16,14.94,7.6,11.7,1,0,B Evans,ZIM
4,N Senamontree (THA),2023-2026,26,26,98.5,10,486,39,4/9,12.46,4.91,15.2,2,0,N Senamontree,THA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1189,V Zanko (CRT),2023-2023,1,1,1.0,0,4,0,-,-,4,-,0,0,V Zanko,CRT
1190,Zawar Farid (UAE),2022-2022,1,1,4.0,0,33,0,-,-,8.25,-,0,0,Zawar Farid,UAE
1191,Ziaurahman Shinwari (AUT),2025-2025,1,1,4.0,0,25,0,-,-,6.25,-,0,0,Ziaurahman Shinwari,AUT
1192,V Zimonjic (SRB),2025-2025,2,2,7.0,0,62,0,-,-,8.85,-,0,0,V Zimonjic,SRB


In [79]:
df9 = df9.drop('Span', axis=1)
df9


,Player,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,AM Kimote (TAN),33,33,102.5,3,691,48,4/21,14.39,6.71,12.8,1,0,AM Kimote,TAN
1,BJ McCarthy (IRE),37,37,133.0,2,1103,45,3/7,24.51,8.29,17.7,0,0,BJ McCarthy,IRE
2,Ehsan Khan (HKG),31,31,114.5,3,731,44,4/28,16.61,6.36,15.6,1,0,Ehsan Khan,HKG
3,B Evans (ZIM),23,23,76.4,0,583,39,4/16,14.94,7.6,11.7,1,0,B Evans,ZIM
4,N Senamontree (THA),26,26,98.5,10,486,39,4/9,12.46,4.91,15.2,2,0,N Senamontree,THA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1189,V Zanko (CRT),1,1,1.0,0,4,0,-,-,4,-,0,0,V Zanko,CRT
1190,Zawar Farid (UAE),1,1,4.0,0,33,0,-,-,8.25,-,0,0,Zawar Farid,UAE
1191,Ziaurahman Shinwari (AUT),1,1,4.0,0,25,0,-,-,6.25,-,0,0,Ziaurahman Shinwari,AUT
1192,V Zimonjic (SRB),2,2,7.0,0,62,0,-,-,8.85,-,0,0,V Zimonjic,SRB


In [80]:
def replace_values(value):
    if value == 'DNB':
      return 0
    elif value == 'TDNB':
      return 0
    elif value == 'absent':
      return 0
    elif value == 'sub':
      return 0
    elif value == '-':
      return 0
    else:
      return value

# apply the replace_values function to all columns in the DataFrame
df9 = df9.map(replace_values)
df9


,Player,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,AM Kimote (TAN),33,33,102.5,3,691,48,4/21,14.39,6.71,12.8,1,0,AM Kimote,TAN
1,BJ McCarthy (IRE),37,37,133.0,2,1103,45,3/7,24.51,8.29,17.7,0,0,BJ McCarthy,IRE
2,Ehsan Khan (HKG),31,31,114.5,3,731,44,4/28,16.61,6.36,15.6,1,0,Ehsan Khan,HKG
3,B Evans (ZIM),23,23,76.4,0,583,39,4/16,14.94,7.6,11.7,1,0,B Evans,ZIM
4,N Senamontree (THA),26,26,98.5,10,486,39,4/9,12.46,4.91,15.2,2,0,N Senamontree,THA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1189,V Zanko (CRT),1,1,1.0,0,4,0,0,0,4,0,0,0,V Zanko,CRT
1190,Zawar Farid (UAE),1,1,4.0,0,33,0,0,0,8.25,0,0,0,Zawar Farid,UAE
1191,Ziaurahman Shinwari (AUT),1,1,4.0,0,25,0,0,0,6.25,0,0,0,Ziaurahman Shinwari,AUT
1192,V Zimonjic (SRB),2,2,7.0,0,62,0,0,0,8.85,0,0,0,V Zimonjic,SRB


In [81]:
df9.to_csv('data/First_change_bowlers.csv')


## Second Change

In [82]:
HEADERS = {'User-Agent': 'CricinfoDataProject/1.0 (contact: arkosaha.ruet@gmail.com)'}
# Define the initial page URL
url = 'https://stats.espncricinfo.com/ci/engine/stats/index.html?bowling_positionmax2=4;bowling_positionmin2=4;bowling_positionval2=bowling_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=wickets;page=1;size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling'

response = requests.get(url, headers=HEADERS)
response = response.content
soup = BeautifulSoup(response, 'html.parser')


In [83]:
# Initialize the page number
page_number = 1
total_pages = int(soup.find_all('td', class_='left')[6].text.split(' ')[6].rstrip())


In [84]:
page_number


1

In [85]:
total_pages


7

In [86]:
data = []
for i in range(1, total_pages+1):
  url = f"https://stats.espncricinfo.com/ci/engine/stats/index.html?bowling_positionmax2=4;bowling_positionmin2=4;bowling_positionval2=bowling_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=wickets;page={i};size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling"
  time.sleep(1.5)
  response = requests.get(url, headers=HEADERS)
  response = response.content
  soup = BeautifulSoup(response, 'html.parser')
  # Find the table element that contains the data we want to extract
  table = soup.select_one('#ciHomeContentlhs > div.pnl650M > table:nth-child(5)')

  # Check if the table exists
  if table is not None:
    # Loop through the rows of the table and extract the data for each player
    for row in table.tbody.find_all('tr'):
      columns = row.find_all('td')
      if len(columns) > 0:
        Player = columns[0].text.strip()
        Span = columns[1].text.strip()
        Matches = columns[2].text.strip()
        Innings = columns[3].text.strip()
        Overs = columns[4].text.strip()
        Maidens = columns[5].text.strip()
        Runs = columns[6].text.strip()
        Wickets = columns[7].text.strip()
        BBI = columns[8].text.strip()
        Average = columns[9].text.strip()
        Economy = columns[10].text.strip()
        SR = columns[11].text.strip()
        Four_Wickets = columns[12].text.strip()
        Five_Wickets = columns[13].text.strip()


        # Add the data for each player to the DataFrame
        data.append({
          'Player': Player,
          'Span': Span,
          'Matches': Matches,
          'Innings': Innings,
          'Overs' : Overs,
          'Maidens': Maidens,
          'Runs': Runs,
          'Wickets' : Wickets,
          'BBI': BBI,
          'Average': Average,
          'Economy': Economy,
          'SR': SR,
          'Four_Wickets' : Four_Wickets,
          'Five_Wickets': Five_Wickets
        })

    # Check if there is a next page by finding the 'Next' button on the page
    next_page_link = soup.select_one('.PaginationLink')
    if next_page_link is not None:
      # If there is a next page, increment the page number and update the URL to the next page
      i += 1
    else:
      # If there is no next page, break out of the loop
      break

# Create a DataFrame from the scraped data
dfx = pd.DataFrame(data)


In [87]:
# Print the DataFrame
dfx.head(10)


,Player,Span,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets
0,Rashid Khan (AFG),2022-2026,31,31,121.0,1,722,52,4/17,13.88,5.96,13.9,3,0
1,PW Hasaranga (SL),2022-2026,27,27,103.0,1,741,45,4/17,16.46,7.19,13.7,2,0
2,IO Okpe (NGA),2022-2025,32,32,92.4,2,553,41,3/12,13.48,5.96,13.5,0,0
3,Virandeep Singh (MAS),2022-2026,20,20,73.2,3,354,40,4/5,8.85,4.82,11,2,0
4,Haris Rauf (PAK),2022-2025,24,24,86.3,2,660,39,4/18,16.92,7.63,13.3,3,0
5,AU Rashid (ENG),2022-2026,24,24,90.4,2,607,38,4/11,15.97,6.69,14.3,2,0
6,AR Ramjani (UGA),2022-2025,19,19,73.0,2,372,36,5/17,10.33,5.09,12.1,1,1
7,Mustafizur Rahman (BAN),2022-2025,31,31,116.0,1,776,35,6/10,22.17,6.68,19.8,0,1
8,CV Varun (IND),2024-2026,15,15,56.0,0,450,35,5/17,12.85,8.03,9.6,1,2
9,H Ssenyondo (UGA),2022-2024,18,18,67.4,4,335,30,4/7,11.16,4.95,13.5,3,0


In [88]:
# Extract player name and country using regular expression
dfx[['Player_Name', 'Country']] = dfx['Player'].str.extract(r'^(.*?) \((.*?)\)')


In [89]:
dfx


,Player,Span,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Rashid Khan (AFG),2022-2026,31,31,121.0,1,722,52,4/17,13.88,5.96,13.9,3,0,Rashid Khan,AFG
1,PW Hasaranga (SL),2022-2026,27,27,103.0,1,741,45,4/17,16.46,7.19,13.7,2,0,PW Hasaranga,SL
2,IO Okpe (NGA),2022-2025,32,32,92.4,2,553,41,3/12,13.48,5.96,13.5,0,0,IO Okpe,NGA
3,Virandeep Singh (MAS),2022-2026,20,20,73.2,3,354,40,4/5,8.85,4.82,11,2,0,Virandeep Singh,MAS
4,Haris Rauf (PAK),2022-2025,24,24,86.3,2,660,39,4/18,16.92,7.63,13.3,3,0,Haris Rauf,PAK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1216,Zafer Durmaz (TKY),2025-2025,1,1,1.0,0,8,0,-,-,8,-,0,0,Zafer Durmaz,TKY
1217,Zahid Khan (SLE),2022-2022,1,1,1.0,0,7,0,-,-,7,-,0,0,Zahid Khan,SLE
1218,Zain Ahmad (Fran),2023-2023,1,1,2.0,0,19,0,-,-,9.5,-,0,0,Zain Ahmad,Fran
1219,N Zimonjic (SRB),2023-2023,1,1,1.0,0,12,0,-,-,12,-,0,0,N Zimonjic,SRB


In [90]:
dfx = dfx.drop('Span', axis=1)
dfx


,Player,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Rashid Khan (AFG),31,31,121.0,1,722,52,4/17,13.88,5.96,13.9,3,0,Rashid Khan,AFG
1,PW Hasaranga (SL),27,27,103.0,1,741,45,4/17,16.46,7.19,13.7,2,0,PW Hasaranga,SL
2,IO Okpe (NGA),32,32,92.4,2,553,41,3/12,13.48,5.96,13.5,0,0,IO Okpe,NGA
3,Virandeep Singh (MAS),20,20,73.2,3,354,40,4/5,8.85,4.82,11,2,0,Virandeep Singh,MAS
4,Haris Rauf (PAK),24,24,86.3,2,660,39,4/18,16.92,7.63,13.3,3,0,Haris Rauf,PAK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1216,Zafer Durmaz (TKY),1,1,1.0,0,8,0,-,-,8,-,0,0,Zafer Durmaz,TKY
1217,Zahid Khan (SLE),1,1,1.0,0,7,0,-,-,7,-,0,0,Zahid Khan,SLE
1218,Zain Ahmad (Fran),1,1,2.0,0,19,0,-,-,9.5,-,0,0,Zain Ahmad,Fran
1219,N Zimonjic (SRB),1,1,1.0,0,12,0,-,-,12,-,0,0,N Zimonjic,SRB


In [91]:
def replace_values(value):
    if value == 'DNB':
      return 0
    elif value == 'TDNB':
      return 0
    elif value == 'absent':
      return 0
    elif value == 'sub':
      return 0
    elif value == '-':
      return 0
    else:
      return value

# apply the replace_values function to all columns in the DataFrame
dfx = dfx.map(replace_values)
dfx


,Player,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Rashid Khan (AFG),31,31,121.0,1,722,52,4/17,13.88,5.96,13.9,3,0,Rashid Khan,AFG
1,PW Hasaranga (SL),27,27,103.0,1,741,45,4/17,16.46,7.19,13.7,2,0,PW Hasaranga,SL
2,IO Okpe (NGA),32,32,92.4,2,553,41,3/12,13.48,5.96,13.5,0,0,IO Okpe,NGA
3,Virandeep Singh (MAS),20,20,73.2,3,354,40,4/5,8.85,4.82,11,2,0,Virandeep Singh,MAS
4,Haris Rauf (PAK),24,24,86.3,2,660,39,4/18,16.92,7.63,13.3,3,0,Haris Rauf,PAK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1216,Zafer Durmaz (TKY),1,1,1.0,0,8,0,0,0,8,0,0,0,Zafer Durmaz,TKY
1217,Zahid Khan (SLE),1,1,1.0,0,7,0,0,0,7,0,0,0,Zahid Khan,SLE
1218,Zain Ahmad (Fran),1,1,2.0,0,19,0,0,0,9.5,0,0,0,Zain Ahmad,Fran
1219,N Zimonjic (SRB),1,1,1.0,0,12,0,0,0,12,0,0,0,N Zimonjic,SRB


In [92]:
dfx.to_csv('data/Second_change_bowlers.csv')


## Others

In [93]:
HEADERS = {'User-Agent': 'CricinfoDataProject/1.0 (contact: arkosaha.ruet@gmail.com)'}
# Define the initial page URL
url = 'https://stats.espncricinfo.com/ci/engine/stats/index.html?bowling_positionmax2=11;bowling_positionmin2=5;bowling_positionval2=bowling_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=wickets;page=1;size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling'

response = requests.get(url, headers=HEADERS)
response = response.content
soup = BeautifulSoup(response, 'html.parser')


In [94]:
# Initialize the page number
page_number = 1
total_pages = int(soup.find_all('td', class_='left')[6].text.split(' ')[6].rstrip())


In [95]:
page_number


1

In [96]:
total_pages


9

In [97]:
data = []
for i in range(1, total_pages+1):
  url = f"https://stats.espncricinfo.com/ci/engine/stats/index.html?bowling_positionmax2=11;bowling_positionmin2=5;bowling_positionval2=bowling_position;class=3;filter=advanced;home_or_away=1;home_or_away=2;home_or_away=3;innings_number=1;innings_number=2;orderby=wickets;page={i};size=200;spanmin1=1+Jan+2022;spanval1=span;template=results;type=bowling"
  time.sleep(1.5)
  response = requests.get(url, headers=HEADERS)
  response = response.content
  soup = BeautifulSoup(response, 'html.parser')
  # Find the table element that contains the data we want to extract
  table = soup.select_one('#ciHomeContentlhs > div.pnl650M > table:nth-child(5)')

  # Check if the table exists
  if table is not None:
    # Loop through the rows of the table and extract the data for each player
    for row in table.tbody.find_all('tr'):
      columns = row.find_all('td')
      if len(columns) > 0:
        Player = columns[0].text.strip()
        Span = columns[1].text.strip()
        Matches = columns[2].text.strip()
        Innings = columns[3].text.strip()
        Overs = columns[4].text.strip()
        Maidens = columns[5].text.strip()
        Runs = columns[6].text.strip()
        Wickets = columns[7].text.strip()
        BBI = columns[8].text.strip()
        Average = columns[9].text.strip()
        Economy = columns[10].text.strip()
        SR = columns[11].text.strip()
        Four_Wickets = columns[12].text.strip()
        Five_Wickets = columns[13].text.strip()


        # Add the data for each player to the DataFrame
        data.append({
          'Player': Player,
          'Span': Span,
          'Matches': Matches,
          'Innings': Innings,
          'Overs' : Overs,
          'Maidens': Maidens,
          'Runs': Runs,
          'Wickets' : Wickets,
          'BBI': BBI,
          'Average': Average,
          'Economy': Economy,
          'SR': SR,
          'Four_Wickets' : Four_Wickets,
          'Five_Wickets': Five_Wickets
        })

    # Check if there is a next page by finding the 'Next' button on the page
    next_page_link = soup.select_one('.PaginationLink')
    if next_page_link is not None:
      # If there is a next page, increment the page number and update the URL to the next page
      i += 1
    else:
      # If there is no next page, break out of the loop
      break

# Create a DataFrame from the scraped data
df0 = pd.DataFrame(data)


In [98]:
# Print the DataFrame
df0.head(10)


,Player,Span,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets
0,Rishad Hossain (BAN),2023-2025,49,49,169.2,1,1398,68,3/18,20.55,8.25,14.9,0,0
1,H Ssenyondo (UGA),2022-2025,43,43,146.4,6,807,68,5/8,11.86,5.5,12.9,3,1
2,Sikandar Raza (ZIM),2022-2026,55,55,184.1,5,1179,65,5/18,18.13,6.4,17,1,1
3,IS Sodhi (NZ),2022-2026,52,52,177.3,0,1435,56,4/28,25.62,8.08,19,1,0
4,A Zampa (AUS),2022-2026,40,40,154.2,0,1218,54,4/21,22.55,7.89,17.1,3,0
5,PW Hasaranga (SL),2022-2026,32,32,124.2,1,924,52,4/15,17.76,7.43,14.3,1,0
6,E Rukiriza (RWN),2023-2025,39,39,122.0,1,840,49,5/11,17.14,6.88,14.9,2,2
7,R Abdulkareem (NGA),2022-2025,37,37,102.4,5,624,48,6/22,13,6.07,12.8,1,1
8,S Lamichhane (NEP),2022-2026,25,25,95.5,1,549,47,5/18,11.68,5.72,12.2,0,1
9,AU Rashid (ENG),2022-2026,44,44,164.0,0,1324,47,3/19,28.17,8.07,20.9,0,0


In [99]:
# Extract player name and country using regular expression
df0[['Player_Name', 'Country']] = df0['Player'].str.extract(r'^(.*?) \((.*?)\)')


In [100]:
df0

,Player,Span,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Rishad Hossain (BAN),2023-2025,49,49,169.2,1,1398,68,3/18,20.55,8.25,14.9,0,0,Rishad Hossain,BAN
1,H Ssenyondo (UGA),2022-2025,43,43,146.4,6,807,68,5/8,11.86,5.5,12.9,3,1,H Ssenyondo,UGA
2,Sikandar Raza (ZIM),2022-2026,55,55,184.1,5,1179,65,5/18,18.13,6.4,17,1,1,Sikandar Raza,ZIM
3,IS Sodhi (NZ),2022-2026,52,52,177.3,0,1435,56,4/28,25.62,8.08,19,1,0,IS Sodhi,NZ
4,A Zampa (AUS),2022-2026,40,40,154.2,0,1218,54,4/21,22.55,7.89,17.1,3,0,A Zampa,AUS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716,Ziaur Rahman (AFG),2026-2026,2,2,6.0,0,57,0,-,-,9.5,-,0,0,Ziaur Rahman,AFG
1717,Ziaur Rehman (FIN),2022-2022,1,1,1.0,0,11,0,-,-,11,-,0,0,Ziaur Rehman,FIN
1718,T Zotos (GRC),2022-2022,1,1,1.0,0,11,0,-,-,11,-,0,0,T Zotos,GRC
1719,Zou Kui (CHN),2024-2024,1,1,2.0,0,37,0,-,-,18.5,-,0,0,Zou Kui,CHN


In [101]:
df0 = df0.drop('Span', axis=1)
df0


,Player,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Rishad Hossain (BAN),49,49,169.2,1,1398,68,3/18,20.55,8.25,14.9,0,0,Rishad Hossain,BAN
1,H Ssenyondo (UGA),43,43,146.4,6,807,68,5/8,11.86,5.5,12.9,3,1,H Ssenyondo,UGA
2,Sikandar Raza (ZIM),55,55,184.1,5,1179,65,5/18,18.13,6.4,17,1,1,Sikandar Raza,ZIM
3,IS Sodhi (NZ),52,52,177.3,0,1435,56,4/28,25.62,8.08,19,1,0,IS Sodhi,NZ
4,A Zampa (AUS),40,40,154.2,0,1218,54,4/21,22.55,7.89,17.1,3,0,A Zampa,AUS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716,Ziaur Rahman (AFG),2,2,6.0,0,57,0,-,-,9.5,-,0,0,Ziaur Rahman,AFG
1717,Ziaur Rehman (FIN),1,1,1.0,0,11,0,-,-,11,-,0,0,Ziaur Rehman,FIN
1718,T Zotos (GRC),1,1,1.0,0,11,0,-,-,11,-,0,0,T Zotos,GRC
1719,Zou Kui (CHN),1,1,2.0,0,37,0,-,-,18.5,-,0,0,Zou Kui,CHN


In [102]:
def replace_values(value):
    if value == 'DNB':
      return 0
    elif value == 'TDNB':
      return 0
    elif value == 'absent':
      return 0
    elif value == 'sub':
      return 0
    elif value == '-':
      return 0
    else:
      return value

# apply the replace_values function to all columns in the DataFrame
df0 = df0.map(replace_values)
df0


,Player,Matches,Innings,Overs,Maidens,Runs,Wickets,BBI,Average,Economy,SR,Four_Wickets,Five_Wickets,Player_Name,Country
0,Rishad Hossain (BAN),49,49,169.2,1,1398,68,3/18,20.55,8.25,14.9,0,0,Rishad Hossain,BAN
1,H Ssenyondo (UGA),43,43,146.4,6,807,68,5/8,11.86,5.5,12.9,3,1,H Ssenyondo,UGA
2,Sikandar Raza (ZIM),55,55,184.1,5,1179,65,5/18,18.13,6.4,17,1,1,Sikandar Raza,ZIM
3,IS Sodhi (NZ),52,52,177.3,0,1435,56,4/28,25.62,8.08,19,1,0,IS Sodhi,NZ
4,A Zampa (AUS),40,40,154.2,0,1218,54,4/21,22.55,7.89,17.1,3,0,A Zampa,AUS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716,Ziaur Rahman (AFG),2,2,6.0,0,57,0,0,0,9.5,0,0,0,Ziaur Rahman,AFG
1717,Ziaur Rehman (FIN),1,1,1.0,0,11,0,0,0,11,0,0,0,Ziaur Rehman,FIN
1718,T Zotos (GRC),1,1,1.0,0,11,0,0,0,11,0,0,0,T Zotos,GRC
1719,Zou Kui (CHN),1,1,2.0,0,37,0,0,0,18.5,0,0,0,Zou Kui,CHN


In [ ]:
df0.to_csv('data/Other_changes_bowlers.csv')


: 

In [6]:
import pandas as pd

dff = pd.read_csv('data/bowlers.xlsx', index_col=0)
dff.head(5)

,Player_Name,Country,Overs,Maidens,Runs,Wickets,Economy,Inns,Opposition,Ground,Start_Date
0,S Yeshey,BHU,4.0,1,7,8,1.75,2,Myanmar,Gelephu,26 Dec 2025
1,Syazrul Idrus,MAS,4.0,1,8,7,2,1,China,Kuala Lumpur,26 Jul 2023
2,Ali Dawood,BHR,4.0,0,19,7,4.75,2,Bhutan,Gelephu,11 Dec 2025
3,H Bharadwaj,SGP,4.0,2,3,6,0.75,1,Mongolia,Bangi,5 Sep 2024
4,MA Baig,MWI,4.0,0,9,6,2.25,2,Rwanda,Benoni,16 Dec 2023


In [7]:
dff.to_csv('data/bowlers.csv')